In [2]:
import os

import torch
from torch.utils.data import DataLoader, Subset

from torchvision.datasets import ImageFolder
from torchvision import transforms

import matplotlib.pyplot as plt
import numpy as np

In [3]:
ruta_dataset = "../dataset_cat"

In [5]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [6]:
dataset = ImageFolder(
    root=ruta_dataset,
    transform=transform
)

In [7]:
print("Número de imágenes:", len(dataset))
print("Clases:", dataset.classes)
print("Etiquetas:", dataset.class_to_idx)

Número de imágenes: 1613
Clases: ['Abyssinian', 'Bengal', 'Bombay', 'British Shorthair', 'Egyptian Mau', 'Maine Coon', 'Persian', 'Ragdoll', 'Siamese', 'Sphynx']
Etiquetas: {'Abyssinian': 0, 'Bengal': 1, 'Bombay': 2, 'British Shorthair': 3, 'Egyptian Mau': 4, 'Maine Coon': 5, 'Persian': 6, 'Ragdoll': 7, 'Siamese': 8, 'Sphynx': 9}


In [8]:
imagen, etiqueta = dataset[0]

print("Forma del tensor:", imagen.shape)
print("Etiqueta numérica:", etiqueta)
print("Clase:", dataset.classes[etiqueta])

Forma del tensor: torch.Size([3, 128, 128])
Etiqueta numérica: 0
Clase: Abyssinian


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

In [10]:
indices = np.arange(len(dataset))
etiquetas = dataset.targets

indices_train, indices_temp = train_test_split(
    indices,
    test_size=0.30,
    stratify=etiquetas,
    random_state=42
)

indices_val, indices_test = train_test_split(
    indices_temp,
    test_size=0.50,
    stratify=np.array(etiquetas)[indices_temp],
    random_state=42
)

In [11]:
train_dataset = Subset(dataset, indices_train)
val_dataset = Subset(dataset, indices_val)
test_dataset = Subset(dataset, indices_test)

In [12]:
print("Entrenamiento:", len(train_dataset))
print("Validación:", len(val_dataset))
print("Test:", len(test_dataset))

Entrenamiento: 1129
Validación: 242
Test: 242


In [13]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [14]:
for imagenes, etiquetas in train_loader:
    print("Forma imágenes:", imagenes.shape)
    print("Forma etiquetas:", etiquetas.shape)
    print("Etiquetas:", etiquetas)
    break

Forma imágenes: torch.Size([32, 3, 128, 128])
Forma etiquetas: torch.Size([32])
Etiquetas: tensor([4, 8, 1, 0, 7, 7, 7, 1, 6, 4, 0, 9, 2, 2, 6, 3, 6, 2, 8, 3, 9, 7, 7, 7,
        9, 5, 9, 4, 1, 7, 5, 8])


In [17]:
import torch.nn as nn
import torch.nn.functional as F

In [20]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(32 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [21]:
model = CNN()

print(model)

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [22]:
imagenes, etiquetas = next(iter(train_loader))

salidas = model(imagenes)

print("Entrada:", imagenes.shape)
print("Salida:", salidas.shape)

Entrada: torch.Size([32, 3, 128, 128])
Salida: torch.Size([32, 10])


In [25]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for imagenes, etiquetas in train_loader:

        optimizer.zero_grad()

        salidas = model(imagenes)

        loss = criterion(salidas, etiquetas)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)


    model.eval()

    val_loss = 0.0
    correctos = 0
    total = 0

    with torch.no_grad():

        for imagenes, etiquetas in val_loader:

            salidas = model(imagenes)

            loss = criterion(salidas, etiquetas)

            val_loss += loss.item()

            _, predicciones = torch.max(salidas, 1)

            total += etiquetas.size(0)

            correctos += (predicciones == etiquetas).sum().item()

    val_loss = val_loss / len(val_loader)

    val_accuracy = correctos / total


    print(
        f"Época {epoch + 1}/{num_epochs} - "
        f"Train Loss: {train_loss:.4f} - "
        f"Val Loss: {val_loss:.4f} - "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

KeyboardInterrupt: 